# Using Parallelism with Machine Learning: The Housing Prices Competition 

## Description of the competition

- The Housing Prices Competition train_dataset consists of various features of residential homes in Ames, Iowa, including both quantitative and categorical variables like the size of the property, the number of rooms, year built, and neighborhood quality.
- It includes a set of 79 explanatory variables describing almost every aspect of the houses, allowing for in-depth analysis.
- *The primary goal* of the competition is to predict **the final price of each home**, in this lab we will use *RandomForests*.
- The models are evaluated on Root Mean Squared Error (RMSE) between the logarithm of the predicted value and the logarithm of the observed sales price, encouraging precise predictions over a range of housing prices.

### File descriptions
- *train.csv*: the training set used to train the model.
- *test.csv*: the test set used to compute the performance of the model.
- *train_data_description.txt*: full description of each column.
### Useful train_data fields

Here's a brief version of what you'll find in the train_data description file.

- *SalePrice*: the property's sale price in dollars. This is the target variable that you're trying to predict.
- *MSSubClass*: The building class
- *MSZoning*: The general zoning classification

Teh train_dataset is acessible here: https://www.kaggle.com/code/dansbecker/random-forests/tutorial

## Read and prepare the train_data
*If you're curious about this the professor can explain it for you*.

In [3]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Load the train_dataset
file_path = 'data/train.csv'
train_data = pd.read_csv(file_path, index_col="Id")

# Columns to be deleted
columns_to_delete = ['MoSold', 'YrSold', 'SaleType', 'SaleCondition', 'Alley', 'FireplaceQu', 'PoolQC', 'Fence', 'MiscFeature']

# Delete the specified columns
train_data_cleaned = train_data.drop(columns=columns_to_delete, axis=1)

# Define the input features (X) and the output (y)
X = train_data_cleaned.drop('SalePrice', axis=1)
y = train_data_cleaned['SalePrice']

# Identify the categorical columns in X
categorical_columns = X.select_dtypes(include=['object']).columns

# Initialize a LabelEncoder for each categorical column
label_encoders = {column: LabelEncoder() for column in categorical_columns}

# Apply Label Encoding to each categorical column
for column in categorical_columns:
    X[column] = label_encoders[column].fit_transform(X[column])

# Display the first few rows of X to confirm the encoding
print(X.head())


    MSSubClass  MSZoning  LotFrontage  LotArea  Street  LotShape  LandContour  \
Id                                                                              
1           60         3         65.0     8450       1         3            3   
2           20         3         80.0     9600       1         3            3   
3           60         3         68.0    11250       1         0            3   
4           70         3         60.0     9550       1         0            3   
5           60         3         84.0    14260       1         0            3   

    Utilities  LotConfig  LandSlope  ...  GarageQual  GarageCond  PavedDrive  \
Id                                   ...                                       
1           0          4          0  ...           4           4           2   
2           0          2          0  ...           4           4           2   
3           0          4          0  ...           4           4           2   
4           0          0        

## Split the Data into training and test

In [4]:
from sklearn.model_selection import train_test_split

# Split the first dataset (X, y) into train and test sets with a 70% - 30% split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.30, random_state=42)

# Fill NaN values in X_train and X_val with the median of the respective columns
X_train_filled = X_train.fillna(X_train.median())
X_val_filled = X_val.fillna(X_val.median())

(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

((1022, 70), (438, 70), (1022,), (438,))

## First RandomForest Model
This is the code for a simple trial.

In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from math import sqrt

# Create a Random Forest model
rf_model = RandomForestRegressor(random_state=42)

# Train the model on the training data
rf_model.fit(X_train_filled, y_train)

# Make predictions on the validation data
y_val_pred_filled = rf_model.predict(X_val_filled)

# Calculate the RMSE on the validation data
rmse_filled = sqrt(mean_squared_error(y_val, y_val_pred_filled))

# Print the RMSE
print(f'RMSE on the validation data: {rmse_filled}')

RMSE on the validation data: 26057.941851126383


### Parameters of Random Forest Model
The three most important parameters that typically have the most impact on the performance of a Random Forest model are:

- *n_estimators*: This parameter specifies the number of trees in the forest. Generally, a higher number of trees increases the performance and makes the predictions more stable, but it also makes the computation slower. Selecting the right number of trees requires balancing between performance and computational efficiency.

- *max_features*: This parameter defines the maximum number of features that are allowed to try in an individual tree. There are several options available for this parameter:

    - *sqrt*: This is commonly used and means that the maximum number of features used at each split is the square root of the total number of features.
    - *log2*: This is another typical option, meaning the log base 2 of the feature count is used.
    - *A specific integer or float*: You can specify an exact number or a proportion of the total.

- *max_depth*: This parameter specifies the maximum depth of each tree. Deeper trees can model more complex patterns, but they also risk overfitting. Limiting the depth of trees can improve the model's generalization and reduce overfitting. It's often useful to set this parameter to a finite value, especially when dealing with a large number of features.

## Finding the best parameters sequentially

In [8]:
import time
import threading
import multiprocessing
from math import sqrt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

# Define parameter ranges
n_estimators_range = [10, 25, 50, 100, 200, 300, 400]
max_features_range = ['sqrt', 'log2', None]
max_depth_range = [1, 2, 5, 10, 20, None]

# Get the number of CPU cores
num_cores = multiprocessing.cpu_count()

# -------------------- SEQUENTIAL EXECUTION --------------------

start_time = time.time()

# Initialize best model tracking
best_rmse = float('inf')
best_mape = float('inf')

for n_estimators in n_estimators_range:
    for max_features in max_features_range:
        for max_depth in max_depth_range:
            rf_model = RandomForestRegressor(
                n_estimators=n_estimators,
                max_features=max_features,
                max_depth=max_depth,
                random_state=42
            )
            rf_model.fit(X_train_filled, y_train)

            y_val_pred = rf_model.predict(X_val_filled)
            rmse = sqrt(mean_squared_error(y_val, y_val_pred))
            mape = mean_absolute_percentage_error(y_val, y_val_pred) * 100
            print(f"Sequential - {n_estimators}, {max_features}, {max_depth}. RMSE: {rmse}, MAPE: {mape}%")

end_time = time.time()
sequential_time = end_time - start_time
print(f"\nSequential Execution Time: {sequential_time:.2f} sec")

# -------------------- THREADING EXECUTION --------------------

start_time = time.time()
threads = []

def train_and_evaluate(n_estimators, max_features, max_depth):
    rf_model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_features=max_features,
        max_depth=max_depth,
        random_state=42
    )
    rf_model.fit(X_train_filled, y_train)

    y_val_pred = rf_model.predict(X_val_filled)
    rmse = sqrt(mean_squared_error(y_val, y_val_pred))
    mape = mean_absolute_percentage_error(y_val, y_val_pred) * 100
    print(f"Threading - {n_estimators}, {max_features}, {max_depth}. RMSE: {rmse}, MAPE: {mape}%")

# Create and start threads
for n_estimators in n_estimators_range:
    for max_features in max_features_range:
        for max_depth in max_depth_range:
            t = threading.Thread(target=train_and_evaluate, args=(n_estimators, max_features, max_depth))
            t.start()
            threads.append(t)

# Wait for threads to complete
for t in threads:
    t.join()

end_time = time.time()
threading_time = end_time - start_time
print(f"\nThreading Execution Time: {threading_time:.2f} sec")

# -------------------- MULTIPROCESSING EXECUTION --------------------

start_time = time.time()

def train_and_evaluate_mp(params):
    n_estimators, max_features, max_depth = params
    rf_model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_features=max_features,
        max_depth=max_depth,
        random_state=42
    )
    rf_model.fit(X_train_filled, y_train)

    y_val_pred = rf_model.predict(X_val_filled)
    rmse = sqrt(mean_squared_error(y_val, y_val_pred))
    mape = mean_absolute_percentage_error(y_val, y_val_pred) * 100
    print(f"Multiprocessing - {n_estimators}, {max_features}, {max_depth}. RMSE: {rmse}, MAPE: {mape}%")

if __name__ == '__main__':
    pool = multiprocessing.Pool(processes=num_cores)
    param_combinations = [(n, f, d) for n in n_estimators_range for f in max_features_range for d in max_depth_range]
    pool.map(train_and_evaluate_mp, param_combinations)
    pool.close()
    pool.join()

    end_time = time.time()
    multiprocessing_time = end_time - start_time
    print(f"\nMultiprocessing Execution Time: {multiprocessing_time:.2f} sec")

# -------------------- SPEEDUP & EFFICIENCY CALCULATIONS --------------------

# Amdahl's Law Speedup
parallel_fraction = 0.9  # Assuming 90% of execution is parallelizable
amdahl_speedup_threading = 1 / ((1 - parallel_fraction) + (parallel_fraction / num_cores))
amdahl_speedup_multiprocessing = 1 / ((1 - parallel_fraction) + (parallel_fraction / num_cores))

# Gustafson's Law Speedup
gustafson_speedup_threading = (1 - parallel_fraction) + (parallel_fraction * num_cores)
gustafson_speedup_multiprocessing = (1 - parallel_fraction) + (parallel_fraction * num_cores)

# Actual Speedups (from measured execution times)
actual_speedup_threading = sequential_time / threading_time
actual_speedup_multiprocessing = sequential_time / multiprocessing_time

# Efficiency Calculation
efficiency_threading = actual_speedup_threading / num_cores
efficiency_multiprocessing = actual_speedup_multiprocessing / num_cores

# Print Speedup & Efficiency Results
print("\n### Speedup & Efficiency Calculations ###")
print(f"Amdahl's Speedup (Threading): {amdahl_speedup_threading:.2f}")
print(f"Amdahl's Speedup (Multiprocessing): {amdahl_speedup_multiprocessing:.2f}")
print(f"Gustafson's Speedup (Threading): {gustafson_speedup_threading:.2f}")
print(f"Gustafson's Speedup (Multiprocessing): {gustafson_speedup_multiprocessing:.2f}")
print(f"Actual Speedup (Threading): {actual_speedup_threading:.2f}")
print(f"Actual Speedup (Multiprocessing): {actual_speedup_multiprocessing:.2f}")
print(f"Efficiency (Threading): {efficiency_threading:.2f}")
print(f"Efficiency (Multiprocessing): {efficiency_multiprocessing:.2f}")

# -------------------- FINAL EXECUTION TIME COMPARISON --------------------

print("\n### Final Execution Time Comparison ###")
print(f"Sequential Execution Time: {sequential_time:.2f} sec")
print(f"Threading Execution Time: {threading_time:.2f} sec")
print(f"Multiprocessing Execution Time: {multiprocessing_time:.2f} sec")

Sequential - 10, sqrt, 1. RMSE: 62936.129262244904, MAPE: 26.860734529882123%
Sequential - 10, sqrt, 2. RMSE: 51806.105176310826, MAPE: 20.52147680573829%
Sequential - 10, sqrt, 5. RMSE: 34018.6946431472, MAPE: 13.846204059816515%
Sequential - 10, sqrt, 10. RMSE: 30158.362616471855, MAPE: 11.1847373574493%
Sequential - 10, sqrt, 20. RMSE: 30593.76147467334, MAPE: 11.148658516638926%
Sequential - 10, sqrt, None. RMSE: 28608.440009540453, MAPE: 11.124252485645316%
Sequential - 10, log2, 1. RMSE: 64442.107292878405, MAPE: 27.58882909655716%
Sequential - 10, log2, 2. RMSE: 51396.530250771764, MAPE: 21.498982900746558%
Sequential - 10, log2, 5. RMSE: 34956.394381827566, MAPE: 13.945831547616713%
Sequential - 10, log2, 10. RMSE: 30136.927641357106, MAPE: 11.160840785213171%
Sequential - 10, log2, 20. RMSE: 30841.310557247474, MAPE: 11.240205002153882%
Sequential - 10, log2, None. RMSE: 29000.351214559596, MAPE: 10.98098439114081%
Sequential - 10, None, 1. RMSE: 57823.30960766656, MAPE: 25.85